# Machine Learning Notes
## Day 35: Handling Missing Data (Advanced) — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Advanced Missing Data Strategies for Feature Engineering  
> **Difficulty:** Intermediate  

---
### Note:
FULLY WORKED SOLUTIONS for every exercise in
**Day35_Handling_Missing_Data_Advanced_Practice_Questions.ipynb**.

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

plt.rcParams['figure.figsize'] = (10, 4)
np.random.seed(42)
print('All libraries imported!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Complete Case Analysis (CCA)

In [ ]:
# ============================================================
# ANSWER 1: CCA — when it works and when it fails
# ============================================================

np.random.seed(0)
n = 500
df = pd.DataFrame({
    'Age':    np.random.randint(18, 65, n).astype(float),
    'Income': np.random.randint(20000, 150000, n).astype(float),
    'Score':  np.random.uniform(300, 900, n),
    'City':   np.random.choice(['Mumbai','Delhi','Pune','Chennai'], n),
    'Churn':  np.random.choice([0, 1], n)
})
mcar_idx_age    = np.random.choice(n, int(n*0.03), replace=False)
mcar_idx_income = np.random.choice(n, int(n*0.03), replace=False)
df.loc[mcar_idx_age,    'Age']    = np.nan
df.loc[mcar_idx_income, 'Income'] = np.nan

# MNAR: high-income people hide their income
mnar_idx = df[df['Income'] > 120000].sample(frac=0.6, random_state=0).index
df.loc[mnar_idx, 'Income'] = np.nan

# 1. Missing count and percentage
print('Missing count:')
print(df.isnull().sum())
print('\nMissing percentage:')
print((df.isnull().sum() / len(df) * 100).round(2))

# 2. CCA on MCAR-only columns
income_before_mean = df['Income'].mean()
df_mcar = df[['Age', 'Score', 'City', 'Churn']].dropna()
print(f'\nCCA (MCAR Age only) — retained: {len(df_mcar)/n*100:.1f}%')

# 3. CCA on full df (includes MNAR Income)
df_full_cca = df.dropna()
print(f'CCA (full df, includes MNAR) — retained: {len(df_full_cca)/n*100:.1f}%')

# 4. Compare mean Income before and after CCA
income_after_mean = df_full_cca['Income'].mean()
print(f'\nIncome mean BEFORE CCA: {income_before_mean:,.0f}')
print(f'Income mean AFTER CCA:  {income_after_mean:,.0f}')
print('\nExplanation: MNAR removed high-income earners who hid their data.')
print('CCA dropped exactly those rows — so the remaining dataset UNDER-represents')
print('high earners. Mean Income dropped because the highest earners are now gone.')
print('This is why CCA is DANGEROUS for MNAR data — it introduces systematic bias.')

---
## Section 2: Arbitrary Value Imputation

In [ ]:
# ============================================================
# ANSWER 2: Arbitrary value imputation
# ============================================================

df2 = pd.DataFrame({
    'Age':    [25, np.nan, 33, 55, np.nan, 47, 38, np.nan],
    'Salary': [50000, 85000, np.nan, 120000, np.nan, 95000, 70000, 60000],
    'City':   ['Mumbai', np.nan, 'Pune', 'Mumbai', 'Delhi', np.nan, 'Pune', 'Mumbai'],
})

# 1. Plot BEFORE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df2['Age'].dropna(), bins=8, color='#1565C0', alpha=0.8)
axes[0].set_title('Age BEFORE imputation')
axes[1].hist(df2['Salary'].dropna(), bins=8, color='#2E7D32', alpha=0.8)
axes[1].set_title('Salary BEFORE imputation')
plt.suptitle('Before Imputation\nAmol Jagtap | amoljagtap3001@gmail.com', fontsize=10)
plt.tight_layout(); plt.show()

# 2-4. Apply arbitrary imputation
df2_imp = df2.copy()
df2_imp['Age'].fillna(-999, inplace=True)
df2_imp['Salary'].fillna(-9999, inplace=True)
df2_imp['City'].fillna('Missing', inplace=True)

# 5. Plot AFTER
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df2_imp['Age'], bins=8, color='#C62828', alpha=0.8)
axes[0].set_title('Age AFTER arbitrary imputation (-999)')
axes[1].hist(df2_imp['Salary'], bins=8, color='#E65100', alpha=0.8)
axes[1].set_title('Salary AFTER arbitrary imputation (-9999)')
plt.suptitle('After Arbitrary Imputation\nAmol Jagtap | amoljagtap3001@gmail.com', fontsize=10)
plt.tight_layout(); plt.show()

print('Result:')
print(df2_imp)
print('\nWhy good for trees: A Decision Tree will split on threshold values.')
print('The -999 value is so extreme, the tree will naturally isolate those')
print('rows in a leaf — effectively learning that -999 = was missing.')
print('\nWhy bad for linear models / KNN: A coefficient for Age multiplied')
print('by -999 creates a huge negative term, completely distorting predictions.')
print('KNN sees -999 as extremely far from all real ages — distances are meaningless.')

---
## Section 3: End-of-Tail Imputation

In [ ]:
# ============================================================
# ANSWER 3: End-of-tail imputation
# ============================================================

np.random.seed(1)
age_data    = pd.Series(np.random.normal(35, 8, 200))
salary_data = pd.Series(np.random.exponential(scale=50000, size=200) + 20000)
age_data.iloc[np.random.choice(200, 30, replace=False)] = np.nan
salary_data.iloc[np.random.choice(200, 30, replace=False)] = np.nan

# 1. Age: normal → mean + 3*std
age_observed = age_data.dropna()
age_fill = age_observed.mean() + 3 * age_observed.std()
age_imputed = age_data.fillna(age_fill)
print(f'Age fill value (mean+3σ): {age_fill:.2f}')

# 2. Salary: skewed → Q3 + 1.5*IQR
salary_observed = salary_data.dropna()
Q1  = salary_observed.quantile(0.25)
Q3  = salary_observed.quantile(0.75)
IQR = Q3 - Q1
salary_fill = Q3 + 1.5 * IQR
salary_imputed = salary_data.fillna(salary_fill)
print(f'Salary fill value (Q3+1.5IQR): {salary_fill:,.0f}')

# 3. Side-by-side plots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].hist(age_data.dropna(), bins=25, color='#1565C0', alpha=0.8)
axes[0,0].set_title('Age — Before'); axes[0,0].axvline(age_fill, color='red', linestyle='--', label=f'fill={age_fill:.1f}')
axes[0,0].legend()
axes[0,1].hist(age_imputed, bins=25, color='#C62828', alpha=0.8)
axes[0,1].set_title('Age — After end-of-tail imputation')
axes[1,0].hist(salary_data.dropna(), bins=25, color='#2E7D32', alpha=0.8)
axes[1,0].set_title('Salary — Before'); axes[1,0].axvline(salary_fill, color='red', linestyle='--', label=f'fill={salary_fill:,.0f}')
axes[1,0].legend()
axes[1,1].hist(salary_imputed, bins=25, color='#E65100', alpha=0.8)
axes[1,1].set_title('Salary — After end-of-tail imputation')
plt.suptitle('End-of-Tail Imputation\nAmol Jagtap | amoljagtap3001@gmail.com', fontsize=11)
plt.tight_layout(); plt.show()

---
## Section 4: Frequent Category Imputation

In [ ]:
# ============================================================
# ANSWER 4: Frequent category imputation
# ============================================================

df4 = pd.DataFrame({
    'Gender':    ['Male','Female','Male',np.nan,'Female','Male',np.nan,'Female','Male','Male'],
    'City':      ['Mumbai','Delhi',np.nan,'Pune','Mumbai',np.nan,'Delhi','Mumbai','Pune',np.nan],
    'Education': ['Graduate','School',np.nan,'Postgraduate','Graduate','School',np.nan,'Graduate',np.nan,'School'],
})

# 1. Print modes
print('Mode (most frequent value) per column:')
for col in df4.columns:
    print(f'  {col}: {df4[col].mode()[0]}')

# 2. Fill with mode manually
df4_manual = df4.copy()
for col in df4_manual.columns:
    df4_manual[col].fillna(df4_manual[col].mode()[0], inplace=True)

# 3. Using SimpleImputer
imp = SimpleImputer(strategy='most_frequent')
df4_sklearn = pd.DataFrame(imp.fit_transform(df4), columns=df4.columns)

print('\nManual imputation result:')
print(df4_manual)
print('\nSklearn imputation result:')
print(df4_sklearn)
print('\nResults match:', (df4_manual.values == df4_sklearn.values).all())

# 4. City value_counts before and after
print('\nCity value_counts BEFORE imputation:')
print(df4['City'].value_counts())
print('\nCity value_counts AFTER imputation:')
print(df4_manual['City'].value_counts())
print('\nMumbai (mode) count increased because NaN rows were filled with Mumbai.')

---
## Section 5: Random Sample Imputation

In [ ]:
# ============================================================
# ANSWER 5: Random sample vs mean imputation
# ============================================================

np.random.seed(42)
original = pd.Series(np.random.gamma(2, 2, 300))
data_with_missing = original.copy()
data_with_missing.iloc[np.random.choice(300, 60, replace=False)] = np.nan

# 1. Mean imputation
mean_imputed = data_with_missing.fillna(data_with_missing.mean())

# 2. Random sample imputation
observed = data_with_missing.dropna()
n_missing = data_with_missing.isnull().sum()
random_fill = observed.sample(n=n_missing, replace=True, random_state=42).values
random_imputed = data_with_missing.copy()
random_imputed.loc[random_imputed.isnull()] = random_fill

# 3. Three histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, color in zip(
    axes,
    [original, mean_imputed, random_imputed],
    ['Original (no missing)', 'Mean Imputation', 'Random Sample Imputation'],
    ['#1565C0', '#C62828', '#2E7D32']
):
    ax.hist(data, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{title}\nMean={data.mean():.2f}, Std={data.std():.2f}')
plt.suptitle('Distribution Preservation Comparison\nAmol Jagtap | amoljagtap3001@gmail.com', fontsize=10)
plt.tight_layout(); plt.show()

# 4. Statistics comparison
print(f'Original       — Mean: {original.mean():.3f}, Std: {original.std():.3f}')
print(f'Mean imputed   — Mean: {mean_imputed.mean():.3f}, Std: {mean_imputed.std():.3f}')
print(f'Random imputed — Mean: {random_imputed.mean():.3f}, Std: {random_imputed.std():.3f}')
print('\nConclusion: Random sample imputation preserves mean AND std much')
print('closer to the original. Mean imputation reduces std (creates a spike')
print('at the mean, compressing the spread).')

---
## Section 6: Missing Indicator

In [ ]:
# ============================================================
# ANSWER 6: Missing indicator vs no indicator — model accuracy
# ============================================================

np.random.seed(7)
n = 300
income = np.random.randint(20000, 200000, n).astype(float)
age    = np.random.randint(22, 65, n).astype(float)
target = (income > 80000).astype(int)

hide_idx = np.where(income > 150000)[0]
income[np.random.choice(hide_idx, len(hide_idx)//2, replace=False)] = np.nan

X = pd.DataFrame({'Income': income, 'Age': age})
y = pd.Series(target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Approach A: median imputation only
pipe_a = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('model',  LogisticRegression())
])
scores_a = cross_val_score(pipe_a, X_train, y_train, cv=5, scoring='accuracy')

# Approach B: median imputation + missing indicator
pipe_b = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('model',  LogisticRegression())
])
scores_b = cross_val_score(pipe_b, X_train, y_train, cv=5, scoring='accuracy')

print(f'Approach A (median only)    — CV mean accuracy: {scores_a.mean():.4f}')
print(f'Approach B (median + flag)  — CV mean accuracy: {scores_b.mean():.4f}')

print('\nWhy B is better:')
print('The income is MNAR — specifically, the HIGHEST earners hid their data.')
print('The fact that income is missing is itself a strong signal that income > 150k.')
print('Without the indicator, the model imputes median income (~110k) for')
print('these high earners — wrong direction entirely.')
print('The indicator column (Income_missing=1) tells the model these rows')
print('are special — allowing it to correctly predict them as high income.')

---
## Section 7: KNN Imputation with Weights

In [ ]:
# ============================================================
# ANSWER 7: KNN uniform vs distance weights
# ============================================================

np.random.seed(10)
height = np.random.uniform(150, 190, 100)
weight = height * 0.45 + np.random.normal(0, 3, 100)
df7 = pd.DataFrame({'Height_cm': height, 'Weight_kg': weight})
missing_idx = [5, 20, 40, 60, 80]
true_weight = df7.loc[missing_idx, 'Weight_kg'].copy()
df7.loc[missing_idx, 'Weight_kg'] = np.nan

# 1. Uniform weights
knn_u = KNNImputer(n_neighbors=5, weights='uniform')
res_u = pd.DataFrame(knn_u.fit_transform(df7), columns=df7.columns)
uni_imputed = res_u.loc[missing_idx, 'Weight_kg']

# 2. Distance weights
knn_d = KNNImputer(n_neighbors=5, weights='distance')
res_d = pd.DataFrame(knn_d.fit_transform(df7), columns=df7.columns)
dist_imputed = res_d.loc[missing_idx, 'Weight_kg']

# 3. Comparison table
comp = pd.DataFrame({
    'True_Weight':     true_weight.values,
    'Uniform_Imputed': uni_imputed.values,
    'Distance_Imputed':dist_imputed.values,
}, index=missing_idx)
comp['Uniform_Error']  = abs(comp['True_Weight'] - comp['Uniform_Imputed'])
comp['Distance_Error'] = abs(comp['True_Weight'] - comp['Distance_Imputed'])
print('Comparison table:')
print(comp.round(3))

# 4. MAE
print(f'\nMAE — Uniform weights:  {comp["Uniform_Error"].mean():.3f}')
print(f'MAE — Distance weights: {comp["Distance_Error"].mean():.3f}')
print('\nDistance weights give more influence to the closest neighbours,')
print('which tend to have the most similar heights and thus the most')
print('accurate weight estimates.')

---
## Section 8: MICE / IterativeImputer with Custom Estimator

In [ ]:
# ============================================================
# ANSWER 8: MICE — BayesianRidge vs RandomForest estimator
# ============================================================

np.random.seed(3)
n = 150
x1 = np.random.uniform(1, 10, n)
x2 = x1**2 + np.random.normal(0, 1, n)
x3 = np.log(x1) * 5 + np.random.normal(0, 0.5, n)
df8 = pd.DataFrame({'X1': x1, 'X2': x2, 'X3': x3})
missing_idx = np.random.choice(n, 25, replace=False)
true_x2 = df8.loc[missing_idx, 'X2'].copy()
df8.loc[missing_idx, 'X2'] = np.nan

# 1. BayesianRidge (default)
mice_br = IterativeImputer(max_iter=10, random_state=42)
res_br = pd.DataFrame(mice_br.fit_transform(df8), columns=df8.columns)
mae_br = abs(true_x2.values - res_br.loc[missing_idx, 'X2'].values).mean()

# 2. RandomForest estimator
mice_rf = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=0),
    max_iter=10, random_state=42)
res_rf = pd.DataFrame(mice_rf.fit_transform(df8), columns=df8.columns)
mae_rf = abs(true_x2.values - res_rf.loc[missing_idx, 'X2'].values).mean()

# 3. Results
print(f'MAE — BayesianRidge (linear): {mae_br:.3f}')
print(f'MAE — RandomForest (non-linear): {mae_rf:.3f}')

# 4. Explanation
print('\nExplanation:')
print('X2 = X1² — a QUADRATIC (non-linear) relationship.')
print('BayesianRidge is a linear model — it tries to fit a straight line')
print('through curved data, producing poor estimates.')
print('RandomForest can learn the non-linear x1² relationship through splits,')
print('so its imputed values are much closer to the true quadratic values.')
print('\nRule: Always match the estimator complexity to the data relationships.')

---
## Section 9: Mini End-to-End Project — Comparing Multiple Strategies

In [ ]:
# ============================================================
# ANSWER 9: Comparing imputation strategies on model accuracy
# ============================================================

np.random.seed(99)
n = 400
raw = pd.DataFrame({
    'Age':    np.random.randint(18, 70, n).astype(float),
    'Income': np.random.randint(20000, 200000, n).astype(float),
    'Score':  np.random.uniform(300, 900, n),
    'Churn':  np.random.choice([0, 1], n, p=[0.7, 0.3])
})
raw.loc[np.random.choice(n, int(n*0.20), replace=False), 'Age']    = np.nan
raw.loc[np.random.choice(n, int(n*0.20), replace=False), 'Income'] = np.nan

X = raw.drop(columns=['Churn'])
y = raw['Churn']

strategies = {
    'Mean imputation': Pipeline([
        ('imp',   SimpleImputer(strategy='mean')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression())
    ]),
    'Median imputation': Pipeline([
        ('imp',   SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression())
    ]),
    'Median + indicator': Pipeline([
        ('imp',   SimpleImputer(strategy='median', add_indicator=True)),
        ('scale', StandardScaler()),
        ('model', LogisticRegression())
    ]),
    'KNN imputation': Pipeline([
        ('imp',   KNNImputer(n_neighbors=5)),
        ('scale', StandardScaler()),
        ('model', LogisticRegression())
    ]),
}

print('Comparing imputation strategies (5-fold CV accuracy):')
print('-' * 55)
results = {}
for name, pipe in strategies.items():
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    results[name] = cv_scores.mean()
    print(f'{name:25s}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

best = max(results, key=results.get)
print(f'\nBest strategy: {best} ({results[best]:.4f})')
print('\nNote: Results may vary — the best strategy depends on WHY')
print('data is missing (MCAR/MAR/MNAR) and the feature correlations.')
print('Always test multiple strategies; never assume one is always best.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Strategy | Best When |
|---|---|
| CCA (drop rows) | < 5% missing, MCAR only — dangerous for MAR/MNAR |
| Arbitrary Value (-999) | Tree-based models; signals missingness through unusual value |
| End-of-Tail | Must preserve distribution; more principled than arbitrary |
| Frequent Category (Mode) | Categorical columns with MCAR missingness |
| Random Sample | Distribution preservation critical; less reproducible |
| Missing Indicator | Always add when MAR/MNAR suspected — combine with imputer |
| KNNImputer (uniform) | Correlated features; all neighbours weighted equally |
| KNNImputer (distance) | Closer neighbours more influential — often more accurate |
| MICE + BayesianRidge | Linear feature relationships; fast |
| MICE + RandomForest | Non-linear relationships; more accurate but slower |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 35 — Handling Missing Data (Advanced)